In [ ]:

%matplotlib inline
%config InlineBackend.figure_format = 'png'

import re
import warnings
from pathlib import Path

import phreeqpy.iphreeqc.phreeqc_dll as phreeqc_mod
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats('png')
matplotlib.rcParams['savefig.format'] = 'png'
matplotlib.rcParams['figure.dpi'] = 160
matplotlib.rcParams['savefig.dpi'] = 200

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,
    "axes.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

print(f">>> matplotlib backend: {matplotlib.get_backend()}")
print(">>> Inline figure format: png")


In [ ]:
# =========================================================
# User configuration
# =========================================================
database_path = r"D:\USGS\IPhreeqcCOM 3.8.6-17100\database\PHREEQC_ThermoddemV1.10_15Dec2020.dat"   # Change this to your database path if needed
out_dir = Path("phreeqpy_titration_outputs")
out_dir.mkdir(exist_ok=True)

cfas_reference_fluid = {
    "stage_label": "CFAS Stage-2 boundary fluid",
    "temp": 240,
    "pressure": 27,
    "pH": 8.0,
    "pe": 0.0,
    "units": "mol/kgw",
    "density": 1,
    "Na": 1.05,
    "Cl": "0.50 charge",
    "K": 1e-3,
    "P(5)": 0.10,
    "Cu(1)": 6.432716002312078e-4,
    "Fe(3)": 3.0560699980732676e-4,
    "S(-2)": 2.397170599757541e-2,
    "Au(3)": 1.0e-4,
    "water": 0.008,
}

simulation_config = {
    "title": "FAAS - Hematite Dissolution Titration at CFAS Solution-2 Conditions",
    "solution": {
        "id": 1,
        "temp": cfas_reference_fluid["temp"],
        "pressure": cfas_reference_fluid["pressure"],
        "pH": cfas_reference_fluid["pH"],
        "pe": cfas_reference_fluid["pe"],
        "redox": "pe",
        "units": cfas_reference_fluid["units"],
        "density": cfas_reference_fluid["density"],
        "species": [
            ("Na", str(cfas_reference_fluid["Na"])),
            ("Cl", str(cfas_reference_fluid["Cl"])),
            ("K", str(cfas_reference_fluid["K"])),
            ("P(5)", str(cfas_reference_fluid["P(5)"])),
            ("Cu(1)", str(cfas_reference_fluid["Cu(1)"])),
            ("Fe(3)", str(cfas_reference_fluid["Fe(3)"])),
            ("S(-2)", str(cfas_reference_fluid["S(-2)"])),
            ("Au(3)", str(cfas_reference_fluid["Au(3)"])),
        ],
        "water": cfas_reference_fluid["water"],
    },
    "reaction_temperature": [cfas_reference_fluid["temp"]],
    "reaction_pressure": [cfas_reference_fluid["pressure"]],
    "equilibrium_phases": [
        ("Pyrite", 0.8, 0),
        ("Chalcopyrite(alpha)", -2.0, 0),
        ("Bornite(alpha)", -1.0, 0),
        ("Magnetite", 0.6, 0),
        ("Chalcocite(alpha)", 1.0, 0),
        ("Au(element)", 0, 0),
        ("Cu(element)", 0, 0),
    ],
    "reaction_components": [
        ("Hematite", "5e-7"),
    ],
    "reaction_total_moles": 1,
    "reaction_steps": 1000,
    "incremental_reactions": True,
}

output_config = {
    "activities": [
        "SO4-2", "Cu(HS)2-", "Au(HS)2-", "AuHS",
        "H+", "OH-", "HS-", "H2S",
        "Au(OH)2-", "Au+", "Au+3", "AuCl",
        "AuCl2-", "AuCl3-2", "AuCl4-", "AuOH",
        "Cu2S(HS)2-2", "CuCl+", "CuCl2", "CuCl2-",
        "CuCl3-", "CuCl3-2", "CuCl4-2", "CuHS",
        "CuCl", "CuOH", "CuOH+", "S-2",
    ],
    "equilibrium_phases": [
        "Magnetite", "Chalcocite(alpha)", "Chalcopyrite(alpha)",
        "Bornite(alpha)", "Au(element)", "Pyrite", "Cu(element)",
    ],
    "saturation_indices": [
        "Hematite", "Magnetite", "Chalcocite(alpha)", "Pyrite",
        "Chalcopyrite(alpha)", "Bornite(alpha)", "Cu(element)", "Au(element)",
    ],
    "totals": [
        "Na", "Cl", "K", "P", "Cu", "Fe", "S", "Au",
        "P(5)", "Cu(1)", "Cu(2)", "Fe(2)", "Fe(3)",
        "S(-2)", "S(6)", "Au(1)", "Au(3)",
    ],
    "molalities": [
        "Fe+2", "Fe+3", "FeOH+", "FeOH+2", "FeCl+",
        "FeCl+2", "FeCl2", "FeCl2+", "Fe(OH)4-",
    ],
    "user_punch": {
        "headings": ["rxn_step", "pH_punched", "logaHS", "logaH2S", "SR_Hematite"],
        "lines": [
            '10 IF (STEP_NO <= 0) THEN GOTO 100',
            '20 PUNCH STEP_NO, -LA("H+"), LA("HS-"), LA("H2S"), SR("Hematite")',
            '100 END',
        ],
    },
}

plot_config = {
    "phase_styles": {
        "Magnetite": {"color": "red", "ls": "-", "lw": 2.0, "label": "Magnetite"},
        "Chalcocite(alpha)": {"color": "green", "ls": "-", "lw": 2.0, "label": "Chalcocite"},
        "Pyrite": {"color": "#E6C200", "ls": "-", "lw": 2.5, "label": "Pyrite"},
        "Chalcopyrite(alpha)": {"color": "darkorange", "ls": "-", "lw": 2.5, "label": "Chalcopyrite"},
        "Bornite(alpha)": {"color": "blue", "ls": "-", "lw": 2.5, "label": "Bornite"},
        "Cu(element)": {"color": "black", "ls": "--", "lw": 2.0, "label": "Cu(element)"},
        "Au(element)": {"color": "goldenrod", "ls": "--", "lw": 2.0, "label": "Au(element)"},
    },
    "activity_groups": [
        ["la_H+", "la_OH-", "la_H2S", "la_HS-", "la_S-2"],
        ["la_Cu(HS)2-", "la_Au(HS)2-", "la_AuHS"],
        ["la_CuCl+", "la_CuCl2", "la_CuCl2-", "la_CuCl3-", "la_CuCl3-2", "la_CuCl4-2"],
        ["la_Au+", "la_Au+3", "la_AuCl", "la_AuCl2-", "la_AuCl3-2", "la_AuCl4-", "la_AuOH"],
        ["la_CuOH", "la_CuOH+", "la_SO4-2"],
    ],
    "si_groups": [
        ["si_Hematite", "si_Magnetite", "si_Pyrite", "si_Chalcopyrite(alpha)"],
        ["si_Bornite(alpha)", "si_Chalcocite(alpha)", "si_Cu(element)", "si_Au(element)"],
    ],
}

print(">>> Configuration loaded.")
print(f">>> Reference fluid: {cfas_reference_fluid['stage_label']}")



In [ ]:
# =========================================================
# PHREEQC input builders
# =========================================================
def build_solution_block(cfg):
    lines = [f"SOLUTION {cfg['id']}"]
    lines.append(f"    temp      {cfg['temp']}")
    if "pressure" in cfg:
        lines.append(f"    pressure  {cfg['pressure']}")
    lines.append(f"    pH        {cfg['pH']}")
    lines.append(f"    pe        {cfg['pe']}")
    lines.append(f"    redox     {cfg['redox']}")
    lines.append(f"    units     {cfg['units']}")
    lines.append(f"    density   {cfg['density']}")
    for name, value in cfg["species"]:
        lines.append(f"    {name:<10} {value}")
    lines.append(f"    -water    {cfg['water']}")
    return "\n".join(lines)

def build_reaction_temperature_block(values, block_id=1):
    lines = [f"REACTION_TEMPERATURE {block_id}"]
    for value in values:
        lines.append(f"    {value}")
    return "\n".join(lines)

def build_reaction_pressure_block(values, block_id=1):
    lines = [f"REACTION_PRESSURE {block_id}"]
    for value in values:
        lines.append(f"    {value}")
    return "\n".join(lines)

def build_equilibrium_phases_block(phases, block_id=1):
    lines = [f"EQUILIBRIUM_PHASES {block_id}"]
    for phase_name, si_value, amount_value in phases:
        lines.append(f"    {phase_name:<22} {si_value:<8} {amount_value}")
    return "\n".join(lines)

def build_reaction_block(components, total_moles, n_steps, block_id=1):
    lines = [f"REACTION {block_id}"]
    for reactant_name, reactant_amount in components:
        lines.append(f"    {reactant_name:<22} {reactant_amount}")
    lines.append(f"    {total_moles} moles in {n_steps} steps")
    return "\n".join(lines)

def build_selected_output_block(cfg, block_id=1):
    lines = [f"SELECTED_OUTPUT {block_id}"]
    lines.extend([
        "    -reset                 false",
        "    -high_precision        true",
        "    -simulation            true",
        "    -state                 true",
        "    -solution              true",
        "    -step                  true",
        "    -reaction              false",
        "    -pH                    true",
        "    -pe                    true",
        "    -temperature           true",
        "    -totals                " + "  ".join(cfg["totals"]),
        "    -molalities            " + "  ".join(cfg["molalities"]),
        "    -activities            " + "  ".join(cfg["activities"]),
        "    -equilibrium_phases    " + "  ".join(cfg["equilibrium_phases"]),
        "    -saturation_indices    " + "  ".join(cfg["saturation_indices"]),
    ])
    return "\n".join(lines)

def build_user_punch_block(cfg, block_id=1):
    lines = [f"USER_PUNCH {block_id}"]
    lines.append("    -headings " + " ".join(cfg["headings"]))
    lines.append("    -start")
    lines.extend(cfg["lines"])
    lines.append("    -end")
    return "\n".join(lines)

def build_phreeqc_input(sim_cfg, out_cfg):
    blocks = [
        f"TITLE {sim_cfg['title']}",
        build_solution_block(sim_cfg["solution"]),
        build_reaction_temperature_block(sim_cfg["reaction_temperature"], 1),
        build_reaction_pressure_block(sim_cfg["reaction_pressure"], 1),
        build_equilibrium_phases_block(sim_cfg["equilibrium_phases"], 1),
        build_reaction_block(
            sim_cfg["reaction_components"],
            sim_cfg["reaction_total_moles"],
            sim_cfg["reaction_steps"],
            1,
        ),
        f"INCREMENTAL_REACTIONS {'true' if sim_cfg['incremental_reactions'] else 'false'}",
        build_selected_output_block(out_cfg, 1),
        build_user_punch_block(out_cfg["user_punch"], 1),
        "SAVE equilibrium_phases 2",
        "SAVE solution 2",
        "END",
    ]
    return "\n\n".join(blocks)

phreeqc_input = build_phreeqc_input(simulation_config, output_config)
print(phreeqc_input[:3000])



In [ ]:
# =========================================================
# Run PHREEQC with PhreeqPy
# =========================================================
def run_phreeqc_simulation(database_path, input_text):
    db_path = Path(database_path).expanduser().resolve()

    print(f">>> Working directory: {Path.cwd()}")
    print(f">>> Database path: {db_path}")

    if not db_path.exists():
        raise FileNotFoundError(
            f"PHREEQC database not found:\n{db_path}\n"
            "Please set database_path to the full absolute path of your .dat database file."
        )

    iphreeqc = phreeqc_mod.IPhreeqc()

    try:
        iphreeqc.load_database(str(db_path))
    except Exception as e:
        raise RuntimeError(
            f"Failed to load PHREEQC database:\n{db_path}\n\nOriginal error:\n{e}"
        )

    try:
        iphreeqc.run_string(input_text)
    except Exception as e:
        err_text = ""
        try:
            err_text = iphreeqc.get_error_string()
        except Exception:
            pass
        raise RuntimeError(
            f"PHREEQC run_string failed.\n\n"
            f"IPhreeqc error:\n{err_text}\n\n"
            f"Original exception:\n{e}"
        )

    selected = iphreeqc.get_selected_output_array()
    if not selected or len(selected) < 2:
        raise RuntimeError("No selected output was returned by PHREEQC.")

    header = selected[0]
    rows = selected[1:]
    df_raw = pd.DataFrame(rows, columns=header)
    return iphreeqc, df_raw

def clean_titration_selected_output(df):
    df = df.copy()

    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass

    if "state" in df.columns:
        df = df[df["state"].astype(str).str.strip().eq("react")].copy()

    if "step" in df.columns:
        df["step"] = pd.to_numeric(df["step"], errors="coerce")
        df = df[df["step"].notna()]
        df = df[df["step"] >= 1].copy()

    if "rxn_step" in df.columns:
        df["rxn_step"] = pd.to_numeric(df["rxn_step"], errors="coerce")
        df = df[df["rxn_step"].notna()]
        df = df[df["rxn_step"] >= 1].copy()
        x_col = "rxn_step"
    else:
        x_col = "step"

    df = df.reset_index(drop=True)
    return df, x_col

iphreeqc, df_raw = run_phreeqc_simulation(database_path, phreeqc_input)
df, x_col = clean_titration_selected_output(df_raw)

csv_raw_path = out_dir / "CFAS_titration_selected_output_raw.csv"
csv_clean_path = out_dir / "CFAS_titration_selected_output_clean.csv"
df_raw.to_csv(csv_raw_path, index=False)
df.to_csv(csv_clean_path, index=False)

print(f">>> Raw selected output shape: {df_raw.shape}")
print(f">>> Cleaned selected output shape: {df.shape}")
print(f">>> CSV saved to: {csv_raw_path.resolve()}")
print(f">>> CSV saved to: {csv_clean_path.resolve()}")
print(f">>> Plot x-axis column: {x_col}")

display(df[[c for c in ['simulation', 'state', 'solution', 'step', 'rxn_step'] if c in df.columns]].head())
display(df.head())


In [ ]:
# =========================================================
# Helper functions for plotting
# =========================================================
def nontrivial_series(series, atol=1e-30):
    if series is None:
        return False
    arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return False
    return np.nanmax(np.abs(arr)) > atol

def _to_log_positive(series):
    y = pd.to_numeric(series, errors="coerce").astype(float)
    y = y.where(np.isfinite(y))
    return y.where(y > 0)

def savefig(fig, filename=None):
    if filename is not None:
        path = out_dir / filename
        fig.savefig(path, bbox_inches="tight")
        print(f">>> Saved figure: {path.resolve()}")
    plt.show()

def resolve_exact_output_names(df):
    colset = set(df.columns)

    totals_map = {}
    for name in output_config["totals"]:
        cand = f"{name}(mol/kgw)"
        if cand in colset:
            totals_map[name] = cand

    mol_map = {}
    for name in output_config["molalities"]:
        cand = f"m_{name}(mol/kgw)"
        if cand in colset:
            mol_map[name] = cand

    act_map = {}
    for name in output_config["activities"]:
        cand = f"la_{name}"
        if cand in colset:
            act_map[name] = cand

    eq_map = {}
    for name in output_config["equilibrium_phases"]:
        if name in colset:
            eq_map[name] = name

    si_map = {}
    for name in output_config["saturation_indices"]:
        cand = f"si_{name}"
        if cand in colset:
            si_map[name] = cand

    return totals_map, mol_map, act_map, eq_map, si_map

def plot_grouped_tracking(df, col_groups, out_prefix, title_prefix, ylabel, x_col="step", y_log=False):
    n_fig = 0
    x = pd.to_numeric(df[x_col], errors="coerce")
    for i, cols in enumerate(col_groups, start=1):
        valid_cols = [c for c in cols if c in df.columns]
        if not valid_cols:
            continue

        fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

        plotted = False
        for col in valid_cols:
            y = pd.to_numeric(df[col], errors="coerce")
            if y_log:
                y = _to_log_positive(y)
            if not nontrivial_series(y):
                continue
            ax.plot(x, y, linewidth=2.0, label=col)
            plotted = True

        if not plotted:
            plt.close(fig)
            continue

        ax.set_title(f"{title_prefix} - Group {i}", fontweight="bold")
        ax.set_xlabel("Reaction Step", fontweight="bold")
        ax.set_ylabel(ylabel, fontweight="bold")
        if y_log:
            ax.set_yscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.65)
        savefig(fig, f"{out_prefix}_group_{i:02d}.png")
        n_fig += 1

    return n_fig

def plot_filled_stack_area(df, phase_col_map, out_png, title, ylabel, x_col="step", au_col="Au(element)"):
    """
    100% stacked area plot using absolute mineral amounts.
    Input columns in phase_col_map should be raw phase columns.

    Features:
    1) 100% stacked area along reaction progress
    2) black boundary lines between phases
    3) optional Au curve on secondary y-axis (log scale)
    """
    if len(phase_col_map) == 0:
        return False

    if x_col in df.columns:
        x = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    else:
        x = np.arange(1, len(df) + 1, dtype=float)

    labels = []
    ys = []

    for label, col in phase_col_map.items():
        if col not in df.columns:
            continue
        y = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        y[~np.isfinite(y)] = 0.0
        y = np.clip(y, 0.0, None)
        ys.append(y)
        labels.append(label)

    if len(ys) == 0:
        return False

    Y = np.vstack(ys)
    colsum = Y.sum(axis=0)

    valid = np.isfinite(colsum) & (colsum > 0)
    if not np.any(valid):
        return False

    Y_fill = np.zeros_like(Y, dtype=float)
    Y_fill[:, valid] = Y[:, valid] / colsum[valid]

    plotted_mask = np.any(Y_fill > 0, axis=1)
    if not np.any(plotted_mask):
        return False
    Y_fill = Y_fill[plotted_mask, :]
    labels = [label for label, keep in zip(labels, plotted_mask) if keep]

    fig, ax = plt.subplots(figsize=(30, 6), constrained_layout=True)

    phase_colors = {
        "Hematite": "gray",
        "Pyrite": "#E6C200",
        "Chalcopyrite(alpha)": "darkorange",
        "Bornite(alpha)": "blue",
        "Arsenopyrite": "saddlebrown",
        "Marcassite": "olive",
        "Magnetite": "red",
        "Chalcocite(alpha)": "green",
        "Clinochlore": "purple",
        "Claudetite": "brown",
        "Orpiment": "#7f7f7f",
        "Realgar": "pink",
        "Pyrrhotite": "cyan",
    }
    fallback_palette = list(plt.get_cmap("tab20").colors)
    colors = [phase_colors.get(label, fallback_palette[i % len(fallback_palette)]) for i, label in enumerate(labels)]

    ax.stackplot(
        x, *Y_fill,
        labels=labels,
        colors=colors,
        alpha=0.95
    )

    cum = np.cumsum(Y_fill, axis=0)
    for i in range(cum.shape[0]):
        ax.plot(
            x,
            cum[i, :],
            linewidth=1.0,
            color="black",
            alpha=0.7
        )

    ax2 = None
    if au_col in df.columns:
        y_au = _to_log_positive(df[au_col]).to_numpy(dtype=float)
        y_au_valid = y_au[np.isfinite(y_au)]
        if y_au_valid.size > 0 and np.nanmax(np.abs(y_au_valid)) > 0:
            ax2 = ax.twinx()
            ax2.plot(
                x,
                y_au,
                linestyle="--",
                linewidth=2.2,
                color="purple",
                label="Au(element)"
            )
            ax2.set_ylabel("Au(element) (log scale)", fontweight="bold", color="purple")
            ax2.set_yscale("log")
            ax2.tick_params(axis="y", colors="purple")

    ax.set_xlabel("Reaction Step", fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.set_title(title, fontweight="bold")
    ax.grid(True, alpha=0.25)

    leg1 = ax.legend(
        loc="lower left",
        frameon=True,
        facecolor="white",
        framealpha=0.65,
        fontsize=9
    )
    ax.add_artist(leg1)

    if ax2 is not None:
        lines2, labels2 = ax2.get_legend_handles_labels()
        if lines2:
            ax2.legend(
                lines2, labels2,
                loc="lower right",
                frameon=True,
                facecolor="white",
                framealpha=0.65,
                fontsize=9
            )

    savefig(fig, out_png)
    return True

def render_titration_combined_figure(df, phase_styles, out_png, x_col="step"):
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(
        4, 1, figsize=(10, 14),
        gridspec_kw={"height_ratios": [3, 1.5, 1.5, 1.8]},
        constrained_layout=True,
        sharex=True
    )

    x = pd.to_numeric(df[x_col], errors="coerce")
    ax1b = ax1.twinx()

    for min_name, style in phase_styles.items():
        if min_name in df.columns:
            y = _to_log_positive(df[min_name])
            if nontrivial_series(y, atol=1e-12):
                ax1.plot(
                    x, y,
                    color=style["color"],
                    linestyle=style["ls"],
                    linewidth=style["lw"],
                    label=style["label"]
                )

    if "Au(element)" in df.columns:
        y_au = _to_log_positive(df["Au(element)"])
        if nontrivial_series(y_au, atol=1e-12):
            ax1b.plot(x, y_au, color="purple", linestyle="--", linewidth=2.0, label="Au(element)")
            ax1b.set_ylabel("Au(element) (log scale)", color="purple", fontweight="bold")
            ax1b.set_yscale("log")
            ax1b.tick_params(axis="y", colors="purple")

    ax1.set_title("Reaction Process: Equilibrium Phases", fontweight="bold")
    ax1.set_ylabel("Phase Amount (log scale)", fontweight="bold")
    ax1.set_yscale("log")
    ax1.grid(True, alpha=0.3)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax1b.get_legend_handles_labels()
    if len(h1) + len(h2) > 0:
        ax1.legend(h1 + h2, l1 + l2, loc="lower left",
                   frameon=True, facecolor="white", framealpha=0.65)

    if "pH" in df.columns:
        ax2.plot(x, pd.to_numeric(df["pH"], errors="coerce"), linewidth=2.0, label="pH")
    if "pe" in df.columns:
        ax2b = ax2.twinx()
        ax2b.plot(x, pd.to_numeric(df["pe"], errors="coerce"), linewidth=2.0, linestyle="--", label="pe")
        ax2b.set_ylabel("pe", fontweight="bold")
        h2a, l2a = ax2.get_legend_handles_labels()
        h2b, l2b = ax2b.get_legend_handles_labels()
        ax2.legend(h2a + h2b, l2a + l2b, loc="lower left",
                   frameon=True, facecolor="white", framealpha=0.65)
    ax2.set_title("pH and pe Evolution", fontweight="bold")
    ax2.set_ylabel("pH", fontweight="bold")
    ax2.grid(True, alpha=0.3)

    sulfur_like = [c for c in ["la_H2S", "la_HS-", "la_S-2", "la_AuHS", "la_Cu(HS)2-", "la_Au(HS)2-"] if c in df.columns]
    for col in sulfur_like:
        ax3.plot(x, pd.to_numeric(df[col], errors="coerce"), linewidth=2.0, label=col)
    ax3.set_title("Selected Log Activities", fontweight="bold")
    ax3.set_ylabel("log activity", fontweight="bold")
    ax3.grid(True, alpha=0.3)
    if sulfur_like:
        ax3.legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.65)

    si_cols = [c for c in [
        "si_Hematite", "si_Magnetite", "si_Pyrite",
        "si_Chalcopyrite(alpha)", "si_Bornite(alpha)",
        "si_Chalcocite(alpha)", "si_Cu(element)", "si_Au(element)"
    ] if c in df.columns]
    for col in si_cols:
        ax4.plot(x, pd.to_numeric(df[col], errors="coerce"), linewidth=2.0, label=col)
    ax4.axhline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax4.set_title("Selected Saturation Indices", fontweight="bold")
    ax4.set_xlabel("Reaction Step", fontweight="bold")
    ax4.set_ylabel("SI", fontweight="bold")
    ax4.grid(True, alpha=0.3)
    if si_cols:
        ax4.legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.65)

    savefig(fig, out_png)

totals_map, mol_map, act_map, eq_map, si_map = resolve_exact_output_names(df)

print(">>> Resolved output columns:")
print("    totals :", list(totals_map.keys()))
print("    mol    :", list(mol_map.keys()))
print("    act    :", list(act_map.keys()))
print("    eq     :", list(eq_map.keys()))
print("    si     :", list(si_map.keys()))


In [ ]:
# =========================================================
# Main plots
# =========================================================
render_titration_combined_figure(
    df,
    plot_config["phase_styles"],
    "CFAS_titration_combined.png",
    x_col=x_col
)

phase_fill_map = {phase: phase for phase in output_config["equilibrium_phases"] if phase in df.columns and phase != "Au(element)"}
plot_filled_stack_area(
    df,
    phase_fill_map,
    "FAAS_titration_Mineral_StackArea_Filled.png",
    "Reaction Process: 100% Stacked Mineral Distribution",
    "Fraction of Total Mineral Amount",
    x_col=x_col,
    au_col="Au(element)"
)

plot_grouped_tracking(
    df,
    plot_config["activity_groups"],
    "CFAS_titration_activities",
    "Activity Evolution",
    "log activity",
    x_col=x_col,
    y_log=False
)

plot_grouped_tracking(
    df,
    plot_config["si_groups"],
    "CFAS_titration_si",
    "Saturation Index Evolution",
    "SI",
    x_col=x_col,
    y_log=False
)

phase_cols = [[c for c in output_config["equilibrium_phases"] if c in df.columns]]
plot_grouped_tracking(
    df,
    phase_cols,
    "CFAS_titration_phases",
    "Equilibrium Phase Evolution",
    "Phase Amount",
    x_col=x_col,
    y_log=True
)


In [ ]:
# =========================================================
# Optional quick checks
# =========================================================
display(df.tail())

print(">>> Output folder:", out_dir.resolve())
print(">>> Plot x-axis column:", x_col)
print(">>> You can now edit:")
print("    simulation_config['reaction_components']")
print("    simulation_config['reaction_total_moles']")
print("    simulation_config['reaction_steps']")
print("    output_config / plot_config")
